In [ ]:
import yfinance as yf
import pandas as pd
 
TICKERS = ["NVDA", "MSFT", "BA", "LMT"]
START, END = "2018-01-01", "2025-12-01"
 
prices = yf.download(TICKERS, start=START, end=END, interval="1mo",
                     auto_adjust=True, progress=False)["Close"][TICKERS].dropna()
prices.index = prices.index.to_period("M").to_timestamp("M")
prices.head()


In [ ]:
# Compute trading days per month using daily data
daily = yf.download(TICKERS, start=START, end=END, interval="1d",
                    auto_adjust=True, progress=False)["Close"]
trading_days = (
    daily.resample("ME")
         .count()
         .iloc[:, 0]
         .rename("trading_days")
)
trading_days.index = trading_days.index.to_period("M").to_timestamp("M")
trading_days = trading_days.reindex(prices.index)
trading_days.head()


## 2. Preprocessing

In [ ]:
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import numpy as np

# --- Train/Test split (80/20, time-based) ---
n = len(prices)
split = int(n * 0.80)

prices_train = prices.iloc[:split]
prices_test  = prices.iloc[split:]

td_train = trading_days.iloc[:split]
td_test  = trading_days.iloc[split:]

print(f"Train: {prices_train.index[0].date()} -> {prices_train.index[-1].date()}  ({len(prices_train)} months)")
print(f"Test:  {prices_test.index[0].date()}  -> {prices_test.index[-1].date()}   ({len(prices_test)} months)")

# Log transform to stabilize variance (fit on train, applied same way to test)
log_train = np.log(prices_train)
log_test  = np.log(prices_test)

# First-difference on log prices (= log returns) to achieve stationarity
# Test uses last train value as lag-1 anchor to avoid leakage
diff_train = log_train.diff().dropna()

log_combined = pd.concat([log_train.iloc[[-1]], log_test])
diff_test    = log_combined.diff().dropna()

print("\nTransformed train sample (log-differenced):")
diff_train.head()


## 3. EDA + Diagnostics

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

COLORS = {"NVDA": "#76b900", "MSFT": "#00a4ef", "BA": "#e4002b", "LMT": "#1c3f6e"}
SECTORS = {"Aerospace/Defense": ["LMT", "BA"], "Technology": ["MSFT", "NVDA"]}

# --- Individual stock price series ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
fig.suptitle("Monthly Adjusted Close Prices", fontsize=14, fontweight="bold")
for ax, ticker in zip(axes.flat, TICKERS):
    ax.plot(prices.index, prices[ticker], color=COLORS[ticker], linewidth=1.5)
    ax.axvline(prices_train.index[-1], color="gray", linestyle="--", linewidth=1, label="train/test split")
    ax.set_title(ticker)
    ax.set_ylabel("Price (USD)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
SHARES = 100  # shares per stock for portfolio valuation

# Sector-level value (100 shares each stock in sector)
aero  = (prices[["LMT", "BA"]]  * SHARES).sum(axis=1)
tech  = (prices[["MSFT", "NVDA"]] * SHARES).sum(axis=1)
total = (prices * SHARES).sum(axis=1)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
fig.suptitle("Portfolio Value: Sector & Total (100 shares each)", fontsize=13, fontweight="bold")

for ax, series, label, color in zip(
    axes,
    [aero, tech, total],
    ["Aerospace/Defense", "Technology", "Total Portfolio"],
    ["steelblue", "darkorange", "purple"]
):
    ax.plot(series.index, series, color=color, linewidth=1.8)
    ax.axvline(prices_train.index[-1], color="gray", linestyle="--", linewidth=1)
    ax.set_title(label)
    ax.set_ylabel("Value (USD)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()


In [ ]:
# Summary statistics on raw prices (full series)
summary = prices.describe().T
summary["cv"] = summary["std"] / summary["mean"]   # coefficient of variation
print("Summary Statistics (raw prices)")
summary.round(2)


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ACF / PACF on log-differenced TRAIN series
fig, axes = plt.subplots(len(TICKERS), 2, figsize=(14, 4 * len(TICKERS)))
fig.suptitle("ACF & PACF — Log-Differenced Train Series", fontsize=13, fontweight="bold")

for i, ticker in enumerate(TICKERS):
    series = diff_train[ticker].dropna()
    plot_acf(series,  lags=20, ax=axes[i, 0], title=f"{ticker} — ACF",  zero=False)
    plot_pacf(series, lags=20, ax=axes[i, 1], title=f"{ticker} — PACF", zero=False, method="ywm")

plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

def stationarity_tests(series, name):
    """Run ADF and KPSS on a series, return a summary dict."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf_stat, adf_p, _, _, adf_cv, _ = adfuller(series.dropna(), autolag="AIC")
        kpss_stat, kpss_p, _, kpss_cv    = kpss(series.dropna(), regression="c", nlags="auto")
    return {
        "Ticker": name,
        "ADF stat": round(adf_stat, 4),
        "ADF p-val": round(adf_p, 4),
        "ADF 5% cv": round(adf_cv["5%"], 4),
        "ADF stationary?": "Yes" if adf_p < 0.05 else "No",
        "KPSS stat": round(kpss_stat, 4),
        "KPSS p-val": round(kpss_p, 4),
        "KPSS 5% cv": round(kpss_cv["5%"], 4),
        "KPSS stationary?": "Yes" if kpss_stat < kpss_cv["5%"] else "No",
    }

rows = []
for ticker in TICKERS:
    rows.append(stationarity_tests(prices_train[ticker],     f"{ticker} (levels)"))
    rows.append(stationarity_tests(diff_train[ticker],       f"{ticker} (log-diff)"))

stat_df = pd.DataFrame(rows).set_index("Ticker")
print("Stationarity Tests (Train Set)")
stat_df


## 4. Hierarchical Time Series (HTS) Forecasting

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings

SHARES = 100
h = len(prices_test)   # forecast horizon = test set length

# Bottom-level order used throughout: LMT, BA, MSFT, NVDA
BL = ['LMT', 'BA', 'MSFT', 'NVDA']

val_train = prices_train[BL] * SHARES
val_test  = prices_test[BL]  * SHARES

# Summing matrix S — maps 4 bottom series to all 7 nodes
# Row order: [Total, Aerospace, Technology, LMT, BA, MSFT, NVDA]
S = np.array([
    [1, 1, 1, 1],   # Total
    [1, 1, 0, 0],   # Aerospace (LMT + BA)
    [0, 0, 1, 1],   # Technology (MSFT + NVDA)
    [1, 0, 0, 0],   # LMT
    [0, 1, 0, 0],   # BA
    [0, 0, 1, 0],   # MSFT
    [0, 0, 0, 1],   # NVDA
], dtype=float)

NODE_NAMES = ['Total', 'Aerospace', 'Technology', 'LMT', 'BA', 'MSFT', 'NVDA']

# Actuals at all 7 nodes over the test period
actuals_bottom = val_test.values
actuals_all    = (S @ actuals_bottom.T).T

print(f'Hierarchy: {len(NODE_NAMES)} nodes | Forecast horizon h = {h} months')
print(f'Summing matrix S shape: {S.shape}')


In [ ]:
def fit_arima_forecast(series, h, order=(1, 1, 1)):
    """Fit ARIMA(1,1,1) and return h-step point forecasts."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model = ARIMA(series, order=order).fit()
    return model.forecast(steps=h).values

# Base forecasts at every node using ARIMA(1,1,1)
aero_train  = val_train[['LMT', 'BA']].sum(axis=1)
tech_train  = val_train[['MSFT', 'NVDA']].sum(axis=1)
total_train = val_train.sum(axis=1)

base_bottom = np.column_stack([
    fit_arima_forecast(val_train['LMT'],  h),
    fit_arima_forecast(val_train['BA'],   h),
    fit_arima_forecast(val_train['MSFT'], h),
    fit_arima_forecast(val_train['NVDA'], h),
])  # shape (h, 4)

base_sector = np.column_stack([
    fit_arima_forecast(aero_train,  h),
    fit_arima_forecast(tech_train,  h),
])  # shape (h, 2)

base_total = fit_arima_forecast(total_train, h).reshape(-1, 1)  # (h, 1)

# Full base forecast matrix — same node order as S rows
base_all = np.hstack([base_total, base_sector, base_bottom])  # (h, 7)

print('Base forecasts (first period, all nodes):')
pd.Series(base_all[0], index=NODE_NAMES).round(0)


In [ ]:
# ── Historical proportions for disaggregation (computed on train only) ────────
hist_props = (val_train / val_train.sum(axis=1).values.reshape(-1, 1)).mean()

# ── 1. Bottom-Up: sum bottom forecasts up through the hierarchy ───────────────
bu_bottom = base_bottom.copy()
bu_all    = (S @ bu_bottom.T).T

# ── 2. Top-Down: disaggregate total forecast using historical proportions ──────
td_bottom = base_total * hist_props[BL].values   # (h,1) * (4,) -> (h,4)
td_all    = (S @ td_bottom.T).T

# ── 3. Middle-Out: sector forecasts are base; aggregate up, disaggregate down ─
aero_props = (val_train[['LMT', 'BA']]
              .div(val_train[['LMT', 'BA']].sum(axis=1), axis=0)
              .mean())
tech_props = (val_train[['MSFT', 'NVDA']]
              .div(val_train[['MSFT', 'NVDA']].sum(axis=1), axis=0)
              .mean())

mo_aero   = base_sector[:, [0]]
mo_tech   = base_sector[:, [1]]
mo_bottom = np.hstack([
    mo_aero * aero_props['LMT'],
    mo_aero * aero_props['BA'],
    mo_tech * tech_props['MSFT'],
    mo_tech * tech_props['NVDA'],
])
mo_all = (S @ mo_bottom.T).T

# ── 4. MinT (OLS): project base forecasts onto the coherent subspace ──────────
# ỹ = S(S'S)^{-1}S'ŷ  — simplest closed-form reconciliation (no covariance needed)
P        = S @ np.linalg.inv(S.T @ S) @ S.T
mint_all = (P @ base_all.T).T

print('Reconciliation complete.')
print('Methods: Bottom-Up | Top-Down | Middle-Out | MinT (OLS)')


In [ ]:
def rmse(a, f): return np.sqrt(np.mean((a - f) ** 2))
def mae(a, f):  return np.mean(np.abs(a - f))
def mape(a, f): return np.mean(np.abs((a - f) / a)) * 100

methods = {
    'Base (incoherent)': base_all,
    'Bottom-Up':         bu_all,
    'Top-Down':          td_all,
    'Middle-Out':        mo_all,
    'MinT (OLS)':        mint_all,
}

rows = []
for method, forecasts in methods.items():
    for i, node in enumerate(NODE_NAMES):
        rows.append({
            'Method': method,
            'Node':   node,
            'RMSE':   round(rmse(actuals_all[:, i], forecasts[:, i]), 0),
            'MAE':    round(mae( actuals_all[:, i], forecasts[:, i]), 0),
            'MAPE':   round(mape(actuals_all[:, i], forecasts[:, i]), 2),
        })

eval_df = pd.DataFrame(rows)
print('RMSE by Node and Reconciliation Method:')
eval_df.pivot_table(index='Node', columns='Method', values='RMSE').round(0)


In [ ]:
forecast_colors = {
    'Base (incoherent)': 'gray',
    'Bottom-Up':         'steelblue',
    'Top-Down':          'darkorange',
    'Middle-Out':        'green',
    'MinT (OLS)':        'crimson',
}

# 4-panel plot: Total, Aerospace, Technology, NVDA
plot_nodes = ['Total', 'Aerospace', 'Technology', 'NVDA']
plot_idx   = [NODE_NAMES.index(n) for n in plot_nodes]
test_idx   = prices_test.index

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('HTS Reconciliation: Forecast vs Actual', fontsize=14, fontweight='bold')

for ax, node, idx in zip(axes.flat, plot_nodes, plot_idx):
    ax.plot(test_idx, actuals_all[:, idx], 'k-', linewidth=2, label='Actual', zorder=5)
    for method, forecasts in methods.items():
        ax.plot(test_idx, forecasts[:, idx],
                linestyle='--', linewidth=1.2,
                color=forecast_colors[method], label=method)
    ax.set_title(node)
    ax.set_ylabel('Value (USD)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# Coherence check: Aero + Tech should equal Total for coherent methods
print('\nCoherence check (Aero + Tech vs Total node, period 1):')
for method, forecasts in methods.items():
    dev = abs((forecasts[0, 1] + forecasts[0, 2]) - forecasts[0, 0])
    print(f'  {method:<22} deviation = ${dev:,.2f}')


## 5. VAR / VEC Modeling

In [ ]:
from statsmodels.tsa.stattools import coint

# Pairs to test (same-sector stocks, log-levels on train set)
pairs = [
    ('LMT', 'BA',   'Aerospace/Defense'),
    ('MSFT', 'NVDA', 'Technology'),
]

print('Engle-Granger Cointegration Test (train set, log-levels)')
print('Null hypothesis: NO cointegration\n')
print(f'{"Pair":<18} {"t-stat":>10} {"p-value":>10} {"5% cv":>10} {"Cointegrated?":>15}')
print('-' * 68)

coint_results = {}
for t1, t2, sector in pairs:
    s1 = np.log(prices_train[t1])
    s2 = np.log(prices_train[t2])
    stat, pval, cvs = coint(s1, s2)
    confirmed = pval < 0.05
    coint_results[(t1, t2)] = confirmed
    label = f'{t1} & {t2}'
    print(f'{label:<18} {stat:>10.4f} {pval:>10.4f} {cvs[1]:>10.4f} {"YES" if confirmed else "NO":>15}')

print('\nDecision:')
for (t1, t2), confirmed in coint_results.items():
    model = 'VEC (VECM)' if confirmed else 'VAR'
    print(f'  {t1} & {t2}: use {model}')


In [ ]:
from statsmodels.tsa.api import VAR

# VAR operates on stationary data — use log-differenced train series
# Pairs: LMT & BA (Aerospace), MSFT & NVDA (Technology)
var_pairs = {
    'Aerospace': ['LMT', 'BA'],
    'Technology': ['MSFT', 'NVDA'],
}

print('Lag Selection via AIC/BIC (log-differenced train series)\n')
best_lags = {}

for sector, tickers in var_pairs.items():
    data = diff_train[tickers].dropna()
    model = VAR(data)
    results = model.select_order(maxlags=12)
    aic_lag = results.aic
    bic_lag = results.bic
    # Use BIC lag (more parsimonious); fall back to 1 if 0 is selected
    chosen = max(bic_lag, 1)
    best_lags[sector] = chosen
    print(f'{sector}: AIC -> lag {aic_lag}  |  BIC -> lag {bic_lag}  |  Using lag {chosen}')

print(best_lags)


In [ ]:
# Fit VAR and produce recursive (expanding-window) forecasts
# At each step t, only data up to t-1 is used — no future predictor leakage

var_forecasts = {}   # {sector: np.array shape (h, 2)}
var_models    = {}   # fitted VAR on full train set (for diagnostics)

for sector, tickers in var_pairs.items():
    lag = best_lags[sector]
    train_data = diff_train[tickers].dropna().values   # (n_train, 2)
    test_data  = diff_test[tickers].values             # (h, 2)
    h_steps    = len(test_data)

    # Recursive 1-step-ahead forecasts on log-diff scale
    history = list(train_data)
    preds_diff = []
    for t in range(h_steps):
        window = np.array(history[-lag:])   # last `lag` observations only
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fitted = VAR(np.array(history)).fit(lag)
        fc = fitted.forecast(window, steps=1)[0]
        preds_diff.append(fc)
        history.append(test_data[t])   # append true value (not forecast)

    preds_diff = np.array(preds_diff)  # (h, 2)

    # Convert log-diff forecasts back to price levels
    # cumsum on log-diffs + last known log-level = log-level forecast
    last_log = np.log(prices_train[tickers].iloc[-1].values)   # (2,)
    log_fc   = last_log + np.cumsum(preds_diff, axis=0)
    price_fc = np.exp(log_fc) * SHARES   # back to portfolio value (100 shares)

    var_forecasts[sector] = price_fc

    # Also fit on full train for diagnostics
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        var_models[sector] = VAR(train_data).fit(lag)

    print(f'{sector} VAR(lag={lag}) fit complete. Forecast shape: {price_fc.shape}')


In [ ]:
from statsmodels.stats.stattools import durbin_watson

print('=== Residual Diagnostics ===\n')

for sector, tickers in var_pairs.items():
    res = var_models[sector]
    lag = best_lags[sector]
    print(f'--- {sector} VAR(lag={lag}) ---')

    # 1. Whiteness: Portmanteau test (H0: residuals are white noise)
    try:
        pt = res.test_whiteness(nlags=min(10, len(diff_train) // 4))
        print(f'  Portmanteau whiteness  stat={pt.test_statistic:.4f}  p={pt.pvalue:.4f}  '
              f'{"PASS (white)" if pt.pvalue > 0.05 else "FAIL (autocorrelation present)"}')
    except Exception as e:
        print(f'  Portmanteau test error: {e}')

    # 2. Stability: all roots of characteristic polynomial inside unit circle
    roots = res.roots
    stable = all(np.abs(r) > 1 for r in roots)   # statsmodels returns inverse roots
    print(f'  Stability (all roots outside unit circle): {"STABLE" if stable else "UNSTABLE"}')
    print(f'  Root moduli: {[round(np.abs(r), 4) for r in roots]}')

    # 3. Durbin-Watson on each equation's residuals
    resid = res.resid
    for i, ticker in enumerate(tickers):
        dw = durbin_watson(resid[:, i])
        print(f'  DW [{ticker}]: {dw:.4f}  (near 2 = no autocorrelation)')

    print()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=False)
fig.suptitle('VAR Forecasts vs Actuals (Test Set, 100 shares)', fontsize=13, fontweight='bold')

test_idx = prices_test.index

for row, (sector, tickers) in enumerate(var_pairs.items()):
    fc = var_forecasts[sector]                          # (h, 2) — portfolio values
    actuals = prices_test[tickers].values * SHARES      # (h, 2)

    for col, ticker in enumerate(tickers):
        ax = axes[row, col]
        ax.plot(test_idx, actuals[:, col], 'k-',  linewidth=2, label='Actual')
        ax.plot(test_idx, fc[:, col],      'r--', linewidth=1.5, label='VAR Forecast')
        ax.set_title(f'{ticker} ({sector})')
        ax.set_ylabel('Value (USD)')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.tick_params(axis='x', rotation=30)
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Accuracy metrics
print('VAR Forecast Accuracy (test set):')
print(f'{"Ticker":<8} {"RMSE":>10} {"MAE":>10} {"MAPE%":>10}')
print('-' * 42)
for sector, tickers in var_pairs.items():
    fc = var_forecasts[sector]
    actuals = prices_test[tickers].values * SHARES
    for i, ticker in enumerate(tickers):
        r = rmse(actuals[:, i], fc[:, i])
        m = mae(actuals[:, i],  fc[:, i])
        mp = mape(actuals[:, i], fc[:, i])
        print(f'{ticker:<8} {r:>10.0f} {m:>10.0f} {mp:>10.2f}')
